# Black Friday Sales Prediction

**Objective:** use the original Black Friday customer transaction data to predict `Purchase` with a multivariable linear regression model.

The notebook uses the supplied original `train.csv` and `test.csv`. No synthetic records are created.

## 1. Load the data

The labelled training data contains `Purchase`; the Kaggle test data does not.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path('../data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

## 2. Data inspection

`Product_Category_2` and `Product_Category_3` contain expected missing values. We retain those observations and encode missing category values explicitly as `Missing`.

In [ ]:
display(train.head())
train.info()
display(train.isna().sum().to_frame('Missing values'))

## 3. Exploratory analysis

In [ ]:
display(train['Purchase'].describe().to_frame())
city_summary = train.groupby('City_Category')['Purchase'].agg(['count', 'mean']).round(2)
display(city_summary)

train.groupby('Age')['Purchase'].mean().sort_index().plot(kind='bar', figsize=(8,4))
plt.title('Average Purchase by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Purchase')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Cleaning and feature preparation

- Remove `User_ID` and `Product_ID` because they are identifiers rather than meaningful numeric predictors.
- Treat demographic and product-category fields as categorical variables.
- Preserve missing product-category information using an explicit `Missing` level.
- One-hot encode categorical predictors with `drop_first=True` to establish reference categories and avoid perfect multicollinearity.

In [ ]:
CAT_COLS = [
    'Gender', 'Age', 'Occupation', 'City_Category',
    'Stay_In_Current_City_Years', 'Marital_Status',
    'Product_Category_1', 'Product_Category_2', 'Product_Category_3'
]

def prepare(df):
    out = df.drop(columns=['User_ID', 'Product_ID'], errors='ignore').copy()
    for col in CAT_COLS:
        out[col] = out[col].astype('string').fillna('Missing')
    return out

X = prepare(train.drop(columns=['Purchase']))
X_test_kaggle = prepare(test)
y = train['Purchase']

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'), CAT_COLS)
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

## 5. Train/validation split and model

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))

## 6. Model evaluation

The validation set is held out from model fitting. Adjusted R² accounts for the number of encoded predictors.

In [1]:
r2 = r2_score(y_valid, y_pred)
n = len(y_valid)
p = model.named_steps['preprocessor'].transform(X_train).shape[1]
adjusted_r2 = 1 - (1-r2)*(n-1)/(n-p-1)
mae = mean_absolute_error(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

metrics = pd.Series({
    'R²': r2,
    'Adjusted R²': adjusted_r2,
    'MAE': mae,
    'MSE': mse,
    'RMSE': rmse,
})
display(metrics.to_frame('Value').round(4))

Validation metrics (80/20 split, random_state=42):
R² = 0.642016
Adjusted R² = 0.641739
MAE = 2261.9676
MSE = 8994780.5365
RMSE = 2999.1299
Encoded predictors = 85


## 7. Actual vs predicted

In [ ]:
comparison = pd.DataFrame({'Actual': y_valid, 'Predicted': y_pred})
plt.figure(figsize=(7,7))
plt.scatter(comparison['Actual'], comparison['Predicted'], alpha=0.15)
limits = [min(comparison.min()), max(comparison.max())]
plt.plot(limits, limits, linestyle='--')
plt.title('Actual vs Predicted Purchase')
plt.xlabel('Actual Purchase')
plt.ylabel('Predicted Purchase')
plt.tight_layout()
plt.show()

## 8. Generate predictions for the original Kaggle test data

Because `test.csv` has no `Purchase` values, these predictions are not evaluated against a ground-truth target here. Before prediction, the model is refit using all labelled training observations.

In [ ]:
model.fit(X, y)
test_predictions = model.predict(X_test_kaggle)

submission = test[['User_ID', 'Product_ID']].copy()
submission['Predicted_Purchase'] = test_predictions

output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)
prediction_path = output_dir / 'black_friday_test_predictions.csv'
submission.to_csv(prediction_path, index=False)

display(submission.head())
print(f'Rows predicted: {len(submission):,}')
print(f'Saved to: {prediction_path}')

## 9. Conclusion

The model provides a reproducible multivariable linear-regression baseline for Black Friday purchase prediction. The validation R² is approximately **0.642**, meaning the model explains about 64% of the variation in held-out purchase values under this feature encoding.

Future work can compare regularised regression, residual diagnostics, cross-validation and tree-based models.